In [ ]:
import os
import sys
from google.colab import drive

drive.mount('/content/drive')

if os.path.exists('/content/Motor_Fault_Detection'):
    os.system('git -C /content/Motor_Fault_Detection pull')
    print("Repo updated")
else:
    os.system('git clone https://github.com/dante-jpg2003/Motor_Fault_Detection.git /content/Motor_Fault_Detection')
    print("Repo cloned")

print("Scripts:", os.listdir('/content/Motor_Fault_Detection/scripts'))
print("Drive mounted:", os.path.exists('/content/drive/MyDrive'))

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'scipy', 'scikit-learn', '-q'])
print("Dependencies installed")

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU — go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 2b — Monitor memory
import psutil
import gc

def print_memory():
    ram = psutil.virtual_memory()
    print(f"RAM: {ram.used/1e9:.1f}/{ram.total/1e9:.1f} GB "
          f"({ram.percent}% used)")
    if torch.cuda.is_available():
        gpu_mem = torch.cuda.memory_allocated() / 1e9
        gpu_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU: {gpu_mem:.1f}/{gpu_total:.1f} GB used")

print("Before loading:")
print_memory()

In [ ]:
os.environ['MOTOR_DATA_DIR'] = (
    '/content/drive/MyDrive/Motor_Fault_Detection/data/raw'
)
os.environ['MOTOR_RESULTS_DIR'] = (
    '/content/drive/MyDrive/Motor_Fault_Detection/results'
)
sys.path.append('/content/Motor_Fault_Detection/scripts')
print("Paths configured")
print("Data dir:", os.environ['MOTOR_DATA_DIR'])
print("Results dir:", os.environ['MOTOR_RESULTS_DIR'])

In [ ]:
from train import train

CONFIG = {
    'window_size'  : 800,
    'batch_size'   : 64,
    'epochs'       : 30,
    'learning_rate': 1e-3,
    'weight_decay' : 1e-4,
    'train_ratio'  : 0.8,
    'random_seed'  : 42,
    'num_classes'  : 6,
    'num_channels' : 9,
}
print_memory()
model, history = train(CONFIG)
print_memory()

In [ ]:
import json
import matplotlib.pyplot as plt

with open(os.path.join(os.environ['MOTOR_RESULTS_DIR'], 
                        'training_history.json')) as f:
    history = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['test_loss'],  label='Test')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['test_acc'],  label='Test')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(os.environ['MOTOR_RESULTS_DIR'], 
                          'figures/training_curves.png'), dpi=150)
plt.show()